In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Ruta de tu dataset con las características ya extraídas
ruta_dataset = '/content/drive/MyDrive/Proyecto/Daily_Sports_Activities/Data/dataset_final_features.csv'

# Cargar el dataset
df = pd.read_csv(ruta_dataset)

print(f"Dimensiones del dataset: {df.shape}")
df.head()

Mounted at /content/drive
Dimensiones del dataset: (15200, 243)


,activity,subject,window_id,T_acc_mean_x,T_acc_mean_y,T_acc_mean_z,T_acc_std_x,T_acc_std_y,T_acc_std_z,T_acc_max_x,...,LL_mag_max_x,LL_mag_max_y,LL_mag_max_z,LL_mag_corr_xy,LL_mag_corr_xz,LL_mag_corr_yz,LL_mag_mag_mean,LL_mag_mag_std,LL_mag_mag_auc,LL_mag_mag_mean_diff
0,1,1,0,8.015509,1.058076,5.553903,0.129444,0.039797,0.191729,8.1605,...,0.74182,0.30267,-0.055365,-0.380922,0.214412,-0.094971,0.800508,0.000745,2.369513,0.000941
1,1,1,1,7.920071,1.126935,5.683707,0.058170,0.026639,0.105984,8.0412,...,0.74320,0.30342,-0.054963,-0.351583,0.448888,-0.306916,0.801040,0.000701,2.371086,0.000761
2,1,1,2,8.001183,1.141395,5.559029,0.095242,0.030720,0.148445,8.1763,...,0.74335,0.30377,-0.054945,-0.231525,0.377849,-0.342104,0.801930,0.000862,2.373707,0.000829
3,1,1,3,7.941989,1.143843,5.658659,0.059360,0.024328,0.094272,8.1160,...,0.74302,0.30397,-0.054711,-0.266598,0.365792,-0.255355,0.802269,0.000731,2.374725,0.000797
4,1,1,4,7.996011,1.138048,5.567233,0.042821,0.021047,0.067826,8.0860,...,0.74316,0.30423,-0.055413,-0.169517,0.621999,-0.270246,0.802356,0.000820,2.375000,0.000938


In [3]:
df_features = df.drop(columns=['window_id'])

In [6]:
import time
import numpy as np
from tempfile import mkdtemp
from shutil import rmtree

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA, PCA
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier

# GRID OPTIMIZADO PARA KPCA (Pégalo en tus libretas)
GRID_KPCA = [
    {
        'reduccion__kernel': ['poly'],
        'reduccion__n_components': [32],
        'reduccion__degree': [2, 3], # Solo transformaciones no lineales
        'reduccion__gamma': [None]
    }
]

def nested_loso(
    datos,
    modelo_base,         # <--- Recibe tu modelo intacto
    grid_modelo,         # <--- Recibe tu diccionario de hiperparámetros intacto
    nombre_reduccion,
    target_col="activity",
    group_col="subject",
    scoring="accuracy",
    average="weighted",
    random_state=42
):
    print(f"--- NESTED LOSO | Reducción: {nombre_reduccion.upper()} ---")

    X = datos.drop(columns=[target_col, group_col])
    y = datos[target_col]
    groups = datos[group_col]

    logo_outer = LeaveOneGroupOut()
    fold = 1

    accuracies, precisions, recalls, f1_scores = [], [], [], []
    start_time_total = time.time()

    # 1. Directorio temporal para la caché del Pipeline
    cachedir = mkdtemp()

    try:
        for train_index, test_index in logo_outer.split(X, y, groups):
            start_time_fold = time.time()

            X_test      = X.iloc[test_index]
            y_test      = y.iloc[test_index]
            sujeto_eval = groups.iloc[test_index].unique()[0]

            X_train_val      = X.iloc[train_index]
            y_train_val      = y.iloc[train_index]
            groups_train_val = groups.iloc[train_index]

            # =====================================================
            # 1. CONSTRUCCIÓN DEL PIPELINE SEGÚN REDUCCIÓN
            # =====================================================
            pasos = [('scaler', StandardScaler())]

            if nombre_reduccion == 'pca':
                pasos.append(('reduccion', PCA(n_components=0.95, random_state=random_state)))
                param_grid_red = {}

            elif nombre_reduccion == 'kpca':
                # FIX: Se añade eigen_solver='randomized' para no colapsar la memoria
                pasos.append(('reduccion', KernelPCA(random_state=random_state, n_jobs=-1, eigen_solver='randomized')))
                param_grid_red = GRID_KPCA

            elif nombre_reduccion == 'bosque':
                selector = RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1)
                pasos.append(('reduccion', SelectFromModel(selector)))
                param_grid_red = {}

            elif nombre_reduccion == 'sin':
                param_grid_red = {}

            else:
                raise ValueError(f"Reducción '{nombre_reduccion}' no implementada.")

            # SE AÑADE TU MODELO AL PIPELINE
            pasos.append(('modelo', modelo_base))

            # FIX: Se aplica la memoria caché al pipeline
            pipeline = Pipeline(pasos, memory=cachedir)

            # =====================================================
            # 2. COMBINAR EL GRID DEL MODELO CON EL GRID DE REDUCCIÓN
            # =====================================================
            if isinstance(param_grid_red, list):
                param_grid = [{**bloque, **grid_modelo} for bloque in param_grid_red]
            else:
                param_grid = {**param_grid_red, **grid_modelo}

            # =====================================================
            # 3. INNER CV + GRID SEARCH
            # =====================================================
            inner_cv = GroupKFold(n_splits=3)

            # ¡Aquí se sigue ejecutando el GridSearchCV con tus parámetros!
            search = GridSearchCV(
                pipeline,
                param_grid,
                cv=inner_cv,
                scoring=scoring,
                n_jobs=-1
            )

            search.fit(X_train_val, y_train_val, groups=groups_train_val)

            mejor_pipeline = search.best_estimator_

            # Extraemos los parámetros limpios para la impresión
            params_limpios = {k.split('__')[-1]: v for k, v in search.best_params_.items()}

            # =====================================================
            # 4. PREDICCIÓN Y MÉTRICAS
            # =====================================================
            y_pred = mejor_pipeline.predict(X_test)

            acc_fold = accuracy_score(y_test, y_pred)
            f1_fold  = f1_score(y_test, y_pred, average=average, zero_division=0)

            accuracies.append(acc_fold)
            precisions.append(precision_score(y_test, y_pred, average=average, zero_division=0))
            recalls.append(recall_score(y_test, y_pred, average=average, zero_division=0))
            f1_scores.append(f1_fold)

            print(
                f"[Iter {fold}] Test Sujeto {sujeto_eval} -> "
                f"Params: {params_limpios} | "
                f"Acc: {acc_fold*100:.2f}% | "
                f"F1: {f1_fold*100:.2f}% | "
                f"Tiempo: {time.time() - start_time_fold:.2f} s"
            )

            fold += 1

    finally:
        # Se limpia la caché obligatoriamente al terminar
        rmtree(cachedir)

    # =========================================================
    # RESULTADOS FINALES
    # =========================================================
    print(f"\n==========================================")
    print(f"Tiempo total LOSO Anidado: {(time.time() - start_time_total) / 60:.2f} min")
    print(f"ACCURACY FINAL : {np.mean(accuracies) * 100:.2f}% (± {np.std(accuracies) * 100:.2f}%)")
    print(f"PRECISION FINAL: {np.mean(precisions) * 100:.2f}%")
    print(f"RECALL FINAL   : {np.mean(recalls) * 100:.2f}%")
    print(f"F1-SCORE FINAL : {np.mean(f1_scores) * 100:.2f}%")

    return {
        "accuracy":  np.mean(accuracies),
        "precision": np.mean(precisions),
        "recall":    np.mean(recalls),
        "f1_score":  np.mean(f1_scores),
    }

In [7]:
from sklearn.neural_network import MLPClassifier

# =========================================================
# 1. DEFINIR EL MODELO BASE
# =========================================================
# De acuerdo a tus ejecuciones previas, mantenemos
# max_iter=500 y early_stopping=True como base del modelo
mi_modelo_mlp = MLPClassifier(
    max_iter=500,
    early_stopping=True,
    random_state=42
)

# =========================================================
# 2. DEFINIR EL GRID DE HIPERPARÁMETROS
# =========================================================
# Reconstruido a partir de las arquitecturas de red
# y funciones de activación que evaluaste anteriormente.
# No olvides el prefijo 'modelo__'
mi_grid_mlp = {
    'modelo__activation': ['relu', 'tanh'],
    'modelo__hidden_layer_sizes': [
        (32,),
        (64,),
        (128,),
        (32, 16),
        (64, 32),
        (128, 64),
        (64, 32, 16)
    ]
}

# =========================================================
# 3. LLAMADO A LA FUNCIÓN LOSO ANIDADA CON KPCA
# =========================================================
resultados_mlp_kpca = nested_loso(
    datos=df_features,
    modelo_base=mi_modelo_mlp,
    grid_modelo=mi_grid_mlp,
    nombre_reduccion='kpca'
)

--- NESTED LOSO | Reducción: KPCA ---
[Iter 1] Test Sujeto 1 -> Params: {'activation': 'tanh', 'hidden_layer_sizes': (64,), 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 32} | Acc: 93.21% | F1: 93.00% | Tiempo: 544.62 s
[Iter 2] Test Sujeto 2 -> Params: {'activation': 'tanh', 'hidden_layer_sizes': (128,), 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 32} | Acc: 92.16% | F1: 90.48% | Tiempo: 563.68 s
[Iter 3] Test Sujeto 3 -> Params: {'activation': 'relu', 'hidden_layer_sizes': (32,), 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 32} | Acc: 96.47% | F1: 96.48% | Tiempo: 527.22 s
[Iter 4] Test Sujeto 4 -> Params: {'activation': 'tanh', 'hidden_layer_sizes': (64,), 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 32} | Acc: 97.16% | F1: 97.17% | Tiempo: 531.40 s
[Iter 5] Test Sujeto 5 -> Params: {'activation': 'relu', 'hidden_layer_sizes': (32,), 'degree': 2, 'gamma': None, 'kernel': 'poly', 'n_components': 32} | Acc: 91.79% | F

In [8]:
nested_loso(datos=df_features, nombre_modelo='perceptron', nombre_reduccion='pca')


--- NESTED LOSO | Modelo: PERCEPTRON | Reducción: PCA ---
[Iter 1] Test Sujeto 1 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (128, 64, 32), 'max_iter': 500, 'n_components': 0.95} | Tiempo: 146.30 s
[Iter 2] Test Sujeto 2 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (128,), 'max_iter': 500, 'n_components': 0.95} | Tiempo: 142.86 s
[Iter 3] Test Sujeto 3 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (128,), 'max_iter': 500, 'n_components': 0.95} | Tiempo: 138.21 s
[Iter 4] Test Sujeto 4 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (128, 64), 'max_iter': 500, 'n_components': 0.95} | Tiempo: 141.42 s
[Iter 5] Test Sujeto 5 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (64, 32), 'max_iter': 500, 'n_components': 0.95} | Tiempo: 133.82 s
[Iter 6] Test Sujeto 6 ->

In [9]:
nested_loso(datos=df_features, nombre_modelo='perceptron', nombre_reduccion='bosque')

--- NESTED LOSO | Modelo: PERCEPTRON | Reducción: BOSQUE ---
[Iter 1] Test Sujeto 1 -> Parámetros óptimos: {'activation': 'tanh', 'early_stopping': True, 'hidden_layer_sizes': (32,), 'max_iter': 500} | Tiempo: 986.57 s
[Iter 2] Test Sujeto 2 -> Parámetros óptimos: {'activation': 'tanh', 'early_stopping': True, 'hidden_layer_sizes': (32,), 'max_iter': 500} | Tiempo: 986.05 s
[Iter 3] Test Sujeto 3 -> Parámetros óptimos: {'activation': 'tanh', 'early_stopping': True, 'hidden_layer_sizes': (64, 32, 16), 'max_iter': 500} | Tiempo: 975.46 s
[Iter 4] Test Sujeto 4 -> Parámetros óptimos: {'activation': 'tanh', 'early_stopping': True, 'hidden_layer_sizes': (64,), 'max_iter': 500} | Tiempo: 981.43 s
[Iter 5] Test Sujeto 5 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (128,), 'max_iter': 500} | Tiempo: 988.14 s
[Iter 6] Test Sujeto 6 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (128,), 'max_iter': 500} |

In [10]:
nested_loso(datos=df_features, nombre_modelo='perceptron', nombre_reduccion='sin')

--- NESTED LOSO | Modelo: PERCEPTRON | Reducción: SIN ---
[Iter 1] Test Sujeto 1 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (32, 16), 'max_iter': 500} | Tiempo: 186.63 s
[Iter 2] Test Sujeto 2 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (32, 16), 'max_iter': 500} | Tiempo: 180.34 s
[Iter 3] Test Sujeto 3 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (128, 64), 'max_iter': 500} | Tiempo: 183.77 s
[Iter 4] Test Sujeto 4 -> Parámetros óptimos: {'activation': 'relu', 'early_stopping': True, 'hidden_layer_sizes': (32, 16), 'max_iter': 500} | Tiempo: 184.34 s
[Iter 5] Test Sujeto 5 -> Parámetros óptimos: {'activation': 'tanh', 'early_stopping': True, 'hidden_layer_sizes': (128,), 'max_iter': 500} | Tiempo: 193.00 s
[Iter 6] Test Sujeto 6 -> Parámetros óptimos: {'activation': 'tanh', 'early_stopping': True, 'hidden_layer_sizes': (128, 64), 'max_iter': 